# Request analysis of the whole final dataset (Qwen2.5-VL-7B)

RealEdit's two tasks for every request of the final dataset (16,110 samples): a simplified instruction (Fig. 19)
and its categories (Fig. 12). RealEdit published its instructions but not its categories, so
(`src/instructions/complete.py`):
- 14,861 RealEdit samples keep RealEdit's instruction; the model only labels the categories (`categorize_v1`);
- 1,249 other samples get both from the model (`analyze_v2`).
Result: `out/instructions/qwen2.5-vl-7b/final.jsonl` (one line per sample: instruction, its source, categories)
and `final.summary.json` (done, left, hours left).

Time: about 3 s per request on 2 x T4 (the 7B model is spread over both GPUs), about 13-14 h in all, so two
sessions. Every run stops by itself after `HOURS` (Kaggle ends a session at 12 h, and a session cut off keeps no
output). Every batch is saved at once.

How to run, and go on in a new session:
1. Input: the dataset `cryandrrich/editjudge`. Settings: GPU T4 x2, internet on. Push the code first.
2. Run it with **Save Version > Save & Run All (Commit)**, so it runs in the background and its output is kept.
3. Next session: **Add Input > Your Work > this notebook's latest version** (its output), then Save & Run All
   again. The setup cell finds the earlier results in /kaggle/input and copies them, and the run skips every
   request already done. Repeat until the summary says `"complete": true`.


In [ ]:
import glob, json, os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "curr"
CLIENT = "qwen2.5-vl-7b"
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

def find_dataset():
    """The input folder that holds pairs.jsonl and images/ (the final dataset)."""
    for pattern in ("/kaggle/input/*/pairs.jsonl", "/kaggle/input/*/*/pairs.jsonl", "/kaggle/input/*/*/*/pairs.jsonl"):
        found = sorted(glob.glob(pattern))
        if found:
            return os.path.dirname(found[0])
    raise SystemExit("no input with pairs.jsonl: add the dataset cryandrrich/editjudge")

def find_previous_results():
    """The earlier output with the most analysed requests (a version of this notebook added as input), or None."""
    best, best_rows = None, 0
    for pattern in ("/kaggle/input/*/out/instructions/" + CLIENT, "/kaggle/input/*/*/out/instructions/" + CLIENT,
                    "/kaggle/input/*/instructions/" + CLIENT):
        for folder in glob.glob(pattern):
            rows = sum(sum(1 for _ in open(path)) for path in glob.glob(f"{folder}/*_v*.jsonl"))
            if rows > best_rows:
                best, best_rows = folder, rows
    return best, best_rows

DATA = find_dataset()
print("dataset:", DATA, "-", sum(1 for _ in open(f"{DATA}/pairs.jsonl")), "pairs")
os.environ["EJ_DATA_DIR"] = DATA           # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT          # all results are written here
os.makedirs(f"{OUT}/instructions/{CLIENT}", exist_ok=True)
os.chdir(CODE)

previous, rows = find_previous_results()
if previous:
    shutil.copytree(previous, f"{OUT}/instructions/{CLIENT}", dirs_exist_ok=True)
    print(f"went on from {previous}: {rows} requests done before")
else:
    print("no earlier results found: starting from the beginning")

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:                                       # optional Kaggle secret HF_TOKEN (Add-ons > Secrets): faster downloads
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    os.environ["HF_HUB_VERBOSITY"] = "error"

In [ ]:
!pip -q install -U "transformers>=4.56" accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run_instructions(*args):
    """python -m src.instructions <args>: prints every line as it comes, and returns the output."""
    process = subprocess.Popen([sys.executable, "-u", "-m", "src.instructions", *map(str, args)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)
    return "".join(lines)

In [ ]:
HOURS = 10              # stop before the next batch after this many hours (Kaggle ends a session at 12 h)
SMOKE = False           # True: 8 requests per pass, to check the run (a few minutes plus the model download)


In [ ]:
output = run_instructions("complete", "--client", CLIENT, "--max-hours", HOURS, *(["--limit", 8] if SMOKE else []))


In [ ]:
summary = json.load(open(f"{OUT}/instructions/{CLIENT}/final.summary.json"))
print(json.dumps(summary, indent=1))
if summary["complete"]:
    print("done: download out/instructions/qwen2.5-vl-7b/final.jsonl")
else:
    print(f"{sum(summary['left'].values())} requests left (about {summary['hours_left']} h): add this version's "
          "output as input and Save & Run All again")